In [1]:
import polars as pl

# Data Loading

In [3]:
lf = pl.scan_csv("claude_checkpoint_2020_2025_recoded.csv")
lf_len = lf.select(pl.len()) \
    .collect() \
    .item()


lf.collect().head(3)

title,url,published_date,content,content_chars,news_site,housing_relevance,housing_relevance_reason,exclude_from_housing_analysis,article_type,policy_announcement_flag,expert_commentary_flag,public_reaction_flag,market_data_flag,public_sentiment_coverage,use_in_public_sentiment_index,topics,topic_reason_by_topic,confidence_by_topic,sentiment_by_topic,sentiment_reason_by_topic,sentiment_basis_by_topic,basis_reason_by_topic,public_sentiment_topics,public_sentiment_by_topic,llm_status,llm_error,llm_raw_response,article_index,topics_original
str,str,str,str,i64,str,str,str,bool,str,bool,bool,bool,bool,f64,bool,str,str,str,str,str,str,str,str,str,str,str,str,i64,str
"""'Doesn't really make a differe…","""https://www.channelnewsasia.co…","""2022-08-08T23:31:00+08:00""","""Affected residents were sent a…",7915,"""cna""","""high""","""The article is substantively a…",false,"""mixed""",true,false,true,false,0.7,true,"""Diminishing Lease, SERS and VE…","""{""Diminishing Lease, SERS and …","""{""Diminishing Lease, SERS and …","""{""Diminishing Lease, SERS and …","""{""Diminishing Lease, SERS and …","""{""Diminishing Lease, SERS and …","""{""Diminishing Lease, SERS and …","""Diminishing Lease, SERS and VE…","""{""Diminishing Lease, SERS and …","""done""",null,"""{ ""housing_relevance"": ""high…",0,"""Diminishing Lease, SERS and VE…"
"""'Fair wear and tear' disputes …","""https://www.channelnewsasia.co…","""2021-07-01T14:08:12+08:00""","""After seven years in Singapore…",8636,"""cna""","""medium""","""The article focuses on rental …",false,"""mixed""",false,true,true,false,0.5,true,"""Private Residential Rental Mar…","""{""Private Residential Rental M…","""{""Private Residential Rental M…","""{""Private Residential Rental M…","""{""Private Residential Rental M…","""{""Private Residential Rental M…","""{""Private Residential Rental M…","""Private Residential Rental Mar…","""{""Private Residential Rental M…","""done""",null,"""{ ""housing_relevance"": ""medi…",1,"""Private Residential Rental Mar…"
"""'Feels like a fan': Tengah hom…","""https://www.channelnewsasia.co…","""2023-10-13T06:00:00+08:00""","""SP Group has assured Tengah re…",7920,"""cna""","""high""","""The article focuses on issues …",false,"""mixed""",false,false,true,false,0.7,true,"""Public Housing Quality, Mainte…","""{""Public Housing Quality, Main…","""{""Public Housing Quality, Main…","""{""Public Housing Quality, Main…","""{""Public Housing Quality, Main…","""{""Public Housing Quality, Main…","""{""Public Housing Quality, Main…","""Public Housing Quality, Mainte…","""{""Public Housing Quality, Main…","""done""",null,"""{ ""housing_relevance"": ""high…",2,"""Public Housing Quality, Mainte…"


## Sanity checks

> verify assertions

In [4]:
# no blank data
len(lf.filter(pl.col("title") == "").collect()) == 0

In [4]:
# track number of rows to exclude
exclude_map = (
    pl.col("exclude_from_housing_analysis")
    | (pl.col("housing_relevance").is_in(["low", "none"]))
    | (pl.col("article_type").is_in(["irrelevant", "advertisement_or_sponsored"]))
)

exclude_these = (
    lf.filter(exclude_map)
    .collect()
)

exclude_len = len(exclude_these)

exclude_these

title,url,published_date,content,content_chars,news_site,housing_relevance,housing_relevance_reason,exclude_from_housing_analysis,article_type,policy_announcement_flag,expert_commentary_flag,public_reaction_flag,market_data_flag,public_sentiment_coverage,use_in_public_sentiment_index,topics,topic_reason_by_topic,confidence_by_topic,sentiment_by_topic,sentiment_reason_by_topic,sentiment_basis_by_topic,basis_reason_by_topic,public_sentiment_topics,public_sentiment_by_topic,llm_status,llm_error,llm_raw_response,article_index,topics_original
str,str,str,str,i64,str,str,str,bool,str,bool,bool,bool,bool,f64,bool,str,str,str,str,str,str,str,str,str,str,str,str,i64,str
"""'No need to be greedy': Co-own…","""https://www.channelnewsasia.co…","""2024-11-23T21:30:00+08:00""","""Madam Sng Mui Hong, 71, and he…",10648,"""cna""","""low""","""The article primarily focuses …",true,"""irrelevant""",false,false,false,false,0.0,false,null,"""{}""","""{}""","""{}""","""{}""","""{}""","""{}""",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""low""…",11,null
"""'Unusually low' water level at…","""https://www.channelnewsasia.co…","""2024-02-08T11:21:00+08:00""","""HDB is looking into the cause …",3010,"""cna""","""low""","""The article primarily discusse…",true,"""irrelevant""",false,false,false,false,0.0,false,null,"""{}""","""{}""","""{}""","""{}""","""{}""","""{}""",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""low""…",19,null
"""1 Science Park Drive to be red…","""https://www.channelnewsasia.co…","""2021-11-15T21:43:00+08:00""","""1 Science Park Drive, located …",4760,"""cna""","""none""","""The article is about commercia…",true,"""irrelevant""",false,false,false,false,0.0,false,null,"""{}""","""{}""","""{}""","""{}""","""{}""","""{}""",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""none…",25,null
"""100 residents evacuated in Bed…","""https://www.channelnewsasia.co…","""2024-06-07T09:17:00+08:00""","""A fire broke out in a Bedok No…",869,"""cna""","""low""","""The article is primarily about…",true,"""irrelevant""",false,false,false,false,0.0,false,null,"""{}""","""{}""","""{}""","""{}""","""{}""","""{}""",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""low""…",31,null
"""35 Gilstead: A District 11 hom…","""https://www.channelnewsasia.co…","""2021-09-16T08:50:35+08:00""","""Thoughtfully-designed premium …",4477,"""cna""","""medium""","""The article is about a private…",false,"""advertisement_or_sponsored""",false,false,false,false,0.0,false,"""Private Residential Prices and…","""{""Private Residential Prices a…","""{""Private Residential Prices a…","""{""Private Residential Prices a…","""{""Private Residential Prices a…","""{""Private Residential Prices a…","""{""Private Residential Prices a…",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""medi…",42,"""Private Residential Prices and…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Tender for historic Moulmein R…","""https://www.straitstimes.com/s…","""2025-12-08 18:35:00""","""Country City Investment was aw…",3676,"""st""","""low""","""The article focuses on a comme…",true,"""irrelevant""",false,true,false,true,0.0,false,null,"""{}""","""{}""","""{}""","""{}""","""{}""","""{}""",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""low""…",3314,null
"""HDB dragon boat team member di…","""https://www.straitstimes.com/s…","""2025-11-17 00:24:50""","""A 49-year-old man from HDB’s d…",2338,"""st""","""none""","""The article is about a tragic …",true,"""irrelevant""",false,false,false,false,0.0,false,null,"""{}""","""{}""","""{}""","""{}""","""{}""","""{}""",null,"""{}""","""done""",null,"""{ ""housing_relevance"": ""none…",3324,null
"""SingLand moves to add condo to…","""https://www.businesstimes.com.…","""2025-12-03 19:37:00""","""Acquisition of S$99-million la…",2580,"""bt""","""low""","""The article primarily focuses …",true,"""irrelevant""",false,false,false,true,0.0,false,null,"""{}""","""{}""","""{}""","""{}"""

# Phase 1: Algorithm to translate new dataset to old base format

In [5]:
COLS_TO_KEEP = [
    'title',
    'url',
    'published_date',
    'content',
    'news_site',
    'topics',
    'sentiment_by_topic',
    'sentiment_reason_by_topic'
]

In [ ]:
all_topics = (
    lf.select(
        pl.col("topics")
        .fill_null("")
        .str.split("|")
        .list.eval(pl.element().str.strip_chars())
        .alias("topics")
    )
    .explode("topics")
    .filter(pl.col("topics").is_not_null() & (pl.col("topics") != ""))
    .unique()
    .sort("topics")
    .collect()
    .get_column("topics")
    .to_list()
)

all_topics

['BTO and New Flat Supply',
 'Commercial and Industrial Property',
 'Cooling Measures and Property Regulations',
 'Diminishing Lease, SERS and VERS',
 'Foreign Buyers and External Demand',
 'Government Housing Policy and Schemes',
 'HDB Eligibility, Grants and Buying Rules',
 'HDB Rental and Interim Housing',
 'HDB Resale Prices and Transactions',
 'Housing Affordability and Financial Stress',
 'Land Supply, GLS and Redevelopment',
 'Mortgage, Interest Rates and Housing Loans',
 'Private Residential Prices and Transactions',
 'Private Residential Rental Market',
 'Public Housing Quality, Maintenance and Estate Living',
 'Seniors and Special Housing Needs',
 'Silver Housing Bonus and Right-sizing',
 'Urban Planning and New Town Development']

In [ ]:
def assert_no_articles_with_missing_titles(lf: pl.LazyFrame) -> None:
    count = (
        lf.filter((pl.col("title").str.strip_chars() == "")
                  | pl.col("title").is_null())
            .select(pl.len())
            .collect()
            .item()
    )
    
    assert count == 0, f"Found {count} articles with blank or null title"


def assert_no_articles_with_missing_published_dates(lf: pl.LazyFrame) -> None:
    count = (
        lf.filter(pl.col("published_date").is_null())
            .select(pl.len())
            .collect()
            .item()
    )

    assert count == 0, f"Found {count} articles with null published_date (unparseable date format?)"


def assert_no_duplicate_article_ids(lf: pl.LazyFrame) -> None:
    count = (
        lf.group_by("article_id")
            .len()
            .filter(pl.col("len") > 1)
            .select(pl.len())
            .collect()
            .item()
    )

    assert count == 0, f"Found {count} duplicated article_id values"


def assert_no_articles_with_empty_topics(lf: pl.LazyFrame) -> None:
    count = (
        lf.filter((pl.col('topics').is_null()) 
                  | (pl.col("topics").str.strip_chars() == ""))
            .select(pl.len())
            .collect()
            .item()
    )
    
    assert count == 0, f"Found {count} articles with blank, null, or empty topics"


def assert_no_articles_with_duplicate_topics(lf: pl.LazyFrame) -> None:
    count = (
        lf.filter(pl.col("topics").list.len() != pl.col("topics").list.unique().list.len())
            .select(pl.len())
            .collect()
            .item()
    )

    assert count == 0, f"Found {count} articles with duplicate topics"
    
    
def assert_no_articles_with_missing_sentiments(lf: pl.LazyFrame, all_topics: list[str]) -> None:
    missing_sentiments = (
        lf.with_columns(
            topics_with_sentiments=pl.concat_list([
                pl.when(pl.col("sentiment_by_topic").struct.field(topic).is_not_null())
                  .then(pl.lit(topic))
                for topic in all_topics
            ]).list.drop_nulls()
        )
        .filter(pl.col("topics").list.sort() 
                != pl.col("topics_with_sentiments").list.sort())
        .select(pl.len())
        .collect()
        .item()
    )

    assert missing_sentiments == 0, f"Found {missing_sentiments} articles with topic/sentiment mismatch"


def assert_no_articles_with_missing_sentiment_reasons(lf: pl.LazyFrame, all_topics: list[str]) -> None:
    missing_reasons = (
        lf.with_columns(
            topics_with_reasons=pl.concat_list([
                pl.when(pl.col("sentiment_reason_by_topic").struct.field(topic).is_not_null())
                  .then(pl.lit(topic))
                for topic in all_topics
            ]).list.drop_nulls()
        )
        .filter(
            pl.col("topics").list.sort() !=
            pl.col("topics_with_reasons").list.sort()
        )
        .select(pl.len())
        .collect()
        .item()
    )
    
    assert missing_reasons == 0, f"Found {missing_reasons} articles with topic/explanation mismatch"
    

In [ ]:
exclude_mask = (
    pl.col("exclude_from_housing_analysis")
    | pl.col("housing_relevance").is_in(["low", "none"])
    | pl.col("article_type").is_in(["irrelevant", "advertisement_or_sponsored"])
)

# filter rows marked as irrelevant
# select only columns needed to be kept
base = lf.filter(~exclude_mask) \
    .select(COLS_TO_KEEP)

# Normalise the published_date field
# the raw export is ISO ("2022-08-08T23:31:00+08:00") but excel-converted
# copies come through as "8/8/2022 23:31" - parse both. only the calendar
# day matters downstream, so truncate to a date; this also keeps article_id
# identical across the two export styles (excel drops the seconds)
cleaned_date = (
    pl.col("published_date")
    .str.replace("T", " ")
    .str.replace(r"\+08:00$", "")
)
base = base.with_columns(
    published_date=pl.coalesce(
        cleaned_date.str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
        cleaned_date.str.strptime(pl.Datetime, "%m/%d/%Y %H:%M", strict=False),
    ).dt.date()
)

# strict=False turns anything unparseable into null - fail loudly here
# instead of letting nulls propagate into article_id / year / quarter_year
assert_no_articles_with_missing_published_dates(base)

# create year and quarter columns
base = base.with_columns(
    year=pl.col("published_date").dt.year(),
    quarter_year=(
        pl.col("published_date").dt.year().cast(pl.String)
        + "Q"
        + pl.col("published_date").dt.quarter().cast(pl.String)
    ),
)

# assert no blank titles as title is a partial key
assert_no_articles_with_missing_titles(base)

#
base = base.with_columns(
    title=pl.col("title").str.strip_chars()
)

# Create article id column which uniquely identifies each article
# title + day + news_site: the same story can run on two sites on the
# same day, so news_site is needed to keep those articles distinct
base = base.with_columns(
    article_id=pl.concat_str(
        [
            pl.col('title'), 
            pl.col("published_date").dt.strftime("%Y-%m-%d"),
            pl.col('news_site'),
        ],
        separator=" | "
    )
)

assert_no_duplicate_article_ids(base)

# ensure the topics column has no nulls or empty strings
# all valid articles need to have at least one topic
assert_no_articles_with_empty_topics(base)

# convert topics to list of strings
base = base.with_columns(
    topics=pl.col('topics').str.split("|")
    .list.eval(pl.element().str.strip_chars())
)

# ensure every topic in the topics column is unique
assert_no_articles_with_duplicate_topics(base)

# convert topic, sentiment_by_topic, sentiment_reason_by_topic to a list of structs

# first generate keys for every topic, topics with no sentiment/explanation marked null
sentiment_score_schema = pl.Struct(
    [pl.Field(topic, pl.Float64) for topic in all_topics]
)
sentiment_reason_schema = pl.Struct(
    [pl.Field(topic, pl.String) for topic in all_topics]
)

base = base.with_columns(
    pl.col("sentiment_by_topic").fill_null("{}")
        .str.json_decode(dtype=sentiment_score_schema),
    pl.col("sentiment_reason_by_topic").fill_null("{}")
        .str.json_decode(dtype=sentiment_reason_schema),
)

# assertions to ensure data integrity
assert_no_articles_with_missing_sentiments(base, all_topics)
assert_no_articles_with_missing_sentiment_reasons(base, all_topics)

# convert topics into a list of structs 
# similar to a list of dictionary containing topic, sentiment_score and explanation
base = base.with_columns(
    topics=pl.concat_list([
        pl.when(pl.col("sentiment_by_topic").struct.field(topic).is_not_null())
        .then(
            pl.struct([
                pl.lit(topic).alias("topic"),
                pl.col("sentiment_by_topic").struct.field(topic).alias("sentiment_score"),
                pl.col("sentiment_reason_by_topic").struct.field(topic).alias("explanation")
            ])
        )
        .otherwise(None)
        for topic in all_topics
    ]).list
    .drop_nulls()
)

# reorder columns
reordered_cols = [
    'article_id',
    'title',
    'url',
    'published_date',
    'year',
    'quarter_year',
    'content',
    'news_site',
    'topics',
]

base = base.select(reordered_cols)

# evaluate the lazy frame
base_res = base.collect()

base_res

## Sanity Check

In [28]:
assert lf_len - exclude_len == len(base_res)
print("Correct number of rows, yay! :)")

# write to parquet

In [22]:
base.sink_parquet("new_base.parquet")
print("done")

done
